Class:  https://anthropic.skilljar.com/claude-with-the-anthropic-api/287755

In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [10]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [ ]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle
    """,
)
response = chat(messages, tools=[web_search_schema])

# --- STEP 2: METADATA INTERCEPTION ---
# If Claude requests the web tool, you must loop it back to complete the generation
if response.stop_reason == "tool_use":
    # Append Claude's request to search to the message chain
    messages.append({"role": "assistant", "content": response.content})
    
    # Extract the tool use details returned by the server
    tool_use_block = [b for b in response.content if b.type == "tool_use"][0]
    
    # CRITICAL: Append the corresponding tool result. 
    # For server-side tools, the backend handles the search data payload natively.
    messages.append({
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool_use_block.id,
            "content": response.content
        }]
    })

    # --- STEP 3: FINAL CITATION GENERATION ---
    response = chat(messages, tools=[web_search_schema])


response

Message(id='msg_011CdunirLPvWwjKZuPo4Ym1', container=None, content=[TextBlock(citations=None, text='The best exercises for gaining leg muscle are compound movements that work multiple muscle groups and allow you to lift heavy weights. Here are the top choices:\n\n## **Best Overall: Barbell Back Squat**\nThe squat is widely considered the king of leg exercises because it targets your quadriceps, hamstrings, glutes, and calves simultaneously. It allows for progressive overload with heavy weights, making it ideal for muscle growth.\n\n## **Other Highly Effective Exercises:**\n\n**1. Romanian Deadlifts**\n- Excellent for hamstrings and glutes\n- Teaches proper hip hinge mechanics\n\n**2. Bulgarian Split Squats**\n- Unilateral exercise that addresses muscle imbalances\n- Great for quads, glutes, and stability\n\n**3. Leg Press**\n- Allows for heavy loading with less technical demand\n- Good for quad development when squats are limited\n\n**4. Walking Lunges**\n- Functional movement pattern\